# 3. 모델링

다양한 분류 알고리즘을 적용하고 성능을 비교합니다.
- 기본 모델: RandomForest, LightGBM, XGBoost
- 하이퍼파라미터 튜닝: Ray Tune + Optuna
- AutoML: AutoGluon (Stacking + Bagging)
- 앙상블: Soft Voting

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

train_data = pd.read_csv('../data/train_processed.csv')
test_raw = pd.read_csv('../data/test_processed.csv')
test_ids = pd.read_csv('../data/test_ids.csv')['ID']

X = train_data.drop('label', axis=1)
y = train_data['label']
print(f"Features: {X.shape[1]}, Samples: {X.shape[0]}")

## 3.1 기본 모델 비교 (Cross Validation)

In [ ]:
import lightgbm as lgb
import xgboost as xgb

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'LightGBM': lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, random_state=42, verbose=-1),
    'XGBoost': xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, random_state=42, use_label_encoder=False, eval_metric='logloss'),
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')
    results[name] = scores.mean()
    print(f"{name}: {scores.mean():.5f} (±{scores.std():.5f})")

## 3.2 하이퍼파라미터 튜닝 (Ray Tune + Optuna)

LightGBM과 XGBoost에 대해 Ray Tune 기반 하이퍼파라미터 탐색을 수행합니다.

In [ ]:
import ray
from ray import tune
from ray.tune.search.optuna import OptunaSearch
from ray.tune.schedulers import ASHAScheduler

if ray.is_initialized():
    ray.shutdown()
ray.init(ignore_reinit_error=True)

train_data_ref = ray.put(X)
target_data_ref = ray.put(y)

In [ ]:
def train_lgbm_tune(config):
    X_train = ray.get(train_data_ref)
    y_train = ray.get(target_data_ref)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for tr_idx, val_idx in skf.split(X_train, y_train):
        model = lgb.LGBMClassifier(**config, random_state=42, verbose=-1)
        model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        pred = model.predict(X_train.iloc[val_idx])
        scores.append(accuracy_score(y_train.iloc[val_idx], pred))
    tune.report(accuracy=np.mean(scores))

lgbm_space = {
    'n_estimators': tune.randint(500, 3000),
    'learning_rate': tune.loguniform(1e-3, 0.1),
    'num_leaves': tune.randint(20, 150),
    'max_depth': tune.randint(3, 15),
    'min_child_samples': tune.randint(10, 100),
    'subsample': tune.uniform(0.5, 1.0),
    'colsample_bytree': tune.uniform(0.5, 1.0),
    'reg_alpha': tune.loguniform(1e-3, 10.0),
    'reg_lambda': tune.loguniform(1e-3, 10.0),
}

analysis_lgbm = tune.run(
    train_lgbm_tune,
    config=lgbm_space,
    num_samples=50,
    search_alg=OptunaSearch(metric='accuracy', mode='max'),
    scheduler=ASHAScheduler(metric='accuracy', mode='max'),
    verbose=1,
)

best_lgbm_config = analysis_lgbm.best_config
print(f"Best LightGBM Accuracy: {analysis_lgbm.best_result['accuracy']:.5f}")
print(f"Best Config: {best_lgbm_config}")

## 3.3 AutoGluon (AutoML)

AutoGluon의 `best_quality` 프리셋으로 자동 스태킹 + 배깅을 적용합니다.  
"왜 이 모델이 더 잘 동작하는가?" → AutoGluon은 다양한 모델을 자동으로 앙상블하여 단일 모델 대비 일반화 성능이 높습니다.

In [ ]:
import torch
from autogluon.tabular import TabularPredictor

USE_GPU = torch.cuda.is_available()
print(f"GPU 사용: {USE_GPU}")

save_path = '../outputs/ag_models'

predictor = TabularPredictor(
    label='label',
    eval_metric='accuracy',
    path=save_path,
    problem_type='binary'
).fit(
    train_data,
    presets='best_quality',
    num_bag_folds=10,
    time_limit=3600 * 2,
    num_gpus=1 if USE_GPU else 0,
)

## 3.4 모델 리더보드 확인 및 Soft Voting 앙상블

In [ ]:
lb = predictor.leaderboard(train_data, silent=True)
lb_sorted = lb.sort_values(by='score_val', ascending=False)
base_models = lb_sorted[~lb_sorted['model'].str.contains('WeightedEnsemble')]

top_models = base_models['model'].head(10).tolist()
print("Top 10 모델:")
print(base_models[['model', 'score_val']].head(10))

In [ ]:
# 상위 3개 모델로 Soft Voting 앙상블
selected_models = top_models[:3]
print(f"앙상블 모델: {selected_models}")

preds = []
for model in selected_models:
    pred_proba = predictor.predict_proba(test_raw, model=model).iloc[:, 1]
    preds.append(pred_proba)

final_proba = sum(preds) / len(preds)

submission = pd.DataFrame({
    'ID': test_ids,
    'label': (final_proba >= 0.5).astype(int)
})
submission.to_csv('../outputs/submission.csv', index=False)
print(f"제출 파일 저장 완료: {submission.shape}")
submission.head()